# Calculating BII at the Ethnologue Polygon Level — additional years

Mirrors [`bii_ethnologue.ipynb`](bii_ethnologue.ipynb) (which already produces the year-2000 BII) but loops over the **post-2000 BII rasters** (years 2005, 2010, 2015, 2020 — all `v2-1-1`) and produces one mean-BII column per year.

Output: wide CSV `ethnologue_bii_byyear.csv` with columns `ID, area_km2, bii_2005, bii_2010, bii_2015, bii_2020`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
from rasterstats import zonal_stats

In [ ]:
# Set base project path
base_path = Path("C:/Users/juami/Dropbox/RAships/2-Folklore-Nathan-Project/EA-Maps-Nathan-project/Measures_work")

poscol_path = base_path / "data" / "raw" / "ethnologue" / "ancestral_characteristics_database_language_level" / "Ethnologue_16_shapefile" / "langa_no_overlap_biggest_clean.shp"

maps_path = base_path / "maps" / "raw"
bii_dir   = maps_path / "BII"

# 2000 is already produced by bii_ethnologue.ipynb → skip it here
YEARS = [2005, 2010, 2015, 2020]
bii_files = {year: bii_dir / f"bii-{year}_v2-1-1.tif" for year in YEARS}

for year, fp in bii_files.items():
    print(f"{year}: {'OK' if fp.exists() else 'MISSING'} — {fp.name}")

In [ ]:
# Load Ethnologue polygons
ethnologue = gpd.read_file(poscol_path)

# Compute polygon area in km² using equal-area projection (EPSG:6933)
ethnologue_proj = ethnologue.to_crs(epsg=6933)
ethnologue["area_km2"] = ethnologue_proj.geometry.area / 1e6

# Reproject to BII raster CRS (use first available raster as reference)
with rasterio.open(bii_files[YEARS[0]]) as ref:
    ref_crs = ref.crs
ethnologue = ethnologue.to_crs(ref_crs)

print(f"Number of features: {len(ethnologue)} | Raster CRS: {ref_crs}")

In [ ]:
# Loop over years and compute mean BII per polygon
for year, fp in bii_files.items():
    if not fp.exists():
        print(f"Skipping {year} — file missing")
        continue
    stats = zonal_stats(ethnologue, str(fp), stats=["mean"], geojson_out=False)
    ethnologue[f"bii_{year}"] = [s["mean"] for s in stats]
    print(f"Done year {year}")

In [ ]:
# Build the output dataframe
year_cols = [f"bii_{y}" for y in YEARS if f"bii_{y}" in ethnologue.columns]
df_bii = ethnologue[["ID", "area_km2"] + year_cols].copy()

print(len(df_bii))
df_bii.head()

In [ ]:
# Quick descriptive table — see the BII trajectory across years
df_bii[year_cols].describe()

In [ ]:
# Quick look at year-over-year deltas (e.g., biodiversity decline 2005 → 2020)
if "bii_2020" in df_bii.columns and "bii_2005" in df_bii.columns:
    df_bii["bii_delta_2005_2020"] = df_bii["bii_2020"] - df_bii["bii_2005"]
    print(df_bii[["bii_2005", "bii_2020", "bii_delta_2005_2020"]].describe())

In [ ]:
# Export to CSV
out_path = bii_dir / "ethnologue_bii_byyear.csv"
df_bii.to_csv(out_path, index=False)
print("Exported", out_path)